# 05 — Clinician-Facing Report Generation and Audit Trail

The role this project targets includes AI-supported clinician reports, guardrails, traceability, auditability, and clinician-review mechanisms. This notebook shows the reporting and governance layer.

The report generator is deterministic and template-based for safety. In a production system, an LLM/RAG component could be used only behind structured inputs, evidence retrieval, privacy controls, and clinician review.

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from proteomics_index.modeling import make_baseline_models, binary_metrics
from proteomics_index.scoring import compute_index_table, linear_model_feature_contributions
from proteomics_index.confidence import missingness_penalty, confidence_score_from_signals, confidence_label, calibration_summary
from proteomics_index.report_generator import make_clinician_report
from proteomics_index.governance import dataframe_hash, make_audit_record, save_json

DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

## 1. Fit model and select a sample for reporting

In [2]:
X_raw = pd.read_csv(DATA_PROCESSED / "X_proteomics.csv")
X = X_raw.set_index("SampleID") if "SampleID" in X_raw.columns else X_raw
y_raw = pd.read_csv(DATA_PROCESSED / "y_severe_outcome.csv")
y = y_raw.set_index("SampleID")["SevereOutcome"].loc[X.index] if "SampleID" in y_raw.columns else y_raw.squeeze("columns")
metadata = pd.read_csv(DATA_PROCESSED / "metadata_model.csv").set_index("SampleID").loc[X.index]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=11)
model = make_baseline_models(k_features=40, random_state=11)["ElasticNet_LogReg"]
model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]
miss = missingness_penalty(X_test)
conf_scores = confidence_score_from_signals(probs, miss)
conf_labels = confidence_label(conf_scores)
index_table = compute_index_table(X_test.index, probs, conf_labels, conf_scores)

example_sample_id = index_table.sort_values("index_score", ascending=False).iloc[0]["sample_id"]
example_index = index_table.set_index("sample_id").loc[example_sample_id].copy()
example_index["sample_id"] = example_sample_id
example_metadata = metadata.loc[example_sample_id]
contrib = linear_model_feature_contributions(model, X_test.loc[example_sample_id], list(X.columns), top_n=8)

print("Selected sample:", example_sample_id)
example_index

Selected sample: S048


/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


probability                                          0.962822
index_score                                                96
risk_band                                                 Red
confidence_label                                         High
confidence_score                                     0.881557
decision_gate       Clinician review required: elevated index
ci_lower                                                 None
ci_upper                                                 None
sample_id                                                S048
Name: S048, dtype: object

## 2. Generate clinician-facing decision-support report

In [3]:
model_summary = binary_metrics(y_test, probs)
model_summary.update({k:v for k,v in calibration_summary(y_test, probs).items() if k != "calibration_bins"})

report_md = make_clinician_report(
    index_row=example_index,
    metadata_row=example_metadata,
    contributions=contrib,
    model_summary=model_summary,
)

report_path = REPORTS / "example_clinician_report.md"
report_path.write_text(report_md, encoding="utf-8")
print(report_md[:2000])

# Proteomic Severity Index — Clinician Decision-Support Report

**Generated:** 2026-07-06 19:45 UTC  
**Sample ID:** S048  
**Intended use:** Research/portfolio demonstration of clinical AI architecture. Not for diagnosis or treatment.

## 1. Index Result

- Index score: **96/100**
- Risk band: **Red**
- Model probability: **0.96**
- Confidence: **High** (0.88)
- Review gate: **Clinician review required: elevated index**

## 2. Sample Context

- Age: 75
- Sex: Male
- BMI: 35.3
- Collection day: 19
- Plate ID: Plate_B

## 3. Top Model Drivers

- **Protein_012**: increases index (relative contribution=0.922). No curated knowledge card available in this demo.
- **Protein_013**: increases index (relative contribution=0.779). No curated knowledge card available in this demo.
- **Protein_001**: increases index (relative contribution=0.697). Synthetic immune activation marker used for demonstration.
- **Protein_018**: increases index (relative contribution=0.455). No curated knowledge card av

## 3. Generate audit record

In [4]:
input_snapshot = pd.DataFrame([X_test.loc[example_sample_id]])
audit = make_audit_record(
    sample_id=example_sample_id,
    model_name="ElasticNet_LogReg_ProteomicSeverityIndex",
    model_version="0.2.0-demo",
    input_hash=dataframe_hash(input_snapshot),
    output=example_index.to_dict(),
    reviewer_required="required" in example_index["decision_gate"].lower(),
)

audit_path = REPORTS / "example_audit_record.json"
save_json(audit, str(audit_path))
print(json.dumps(audit, indent=2)[:2000])

{
  "timestamp_utc": "2026-07-06T19:45:27.506914+00:00",
  "sample_id": "S048",
  "model_name": "ElasticNet_LogReg_ProteomicSeverityIndex",
  "model_version": "0.2.0-demo",
  "input_hash_sha256": "62bf006ce0774e2ee58bd061df7a9e065d62e0b4d9c0403a507c6403594271c5",
  "output": {
    "probability": 0.9628217918571029,
    "index_score": 96,
    "risk_band": "Red",
    "confidence_label": "High",
    "confidence_score": 0.881557250235877,
    "decision_gate": "Clinician review required: elevated index",
    "ci_lower": null,
    "ci_upper": null,
    "sample_id": "S048"
  },
  "reviewer_required": true,
  "clinical_use_status": "research_demo_not_for_clinical_use"
}


## 4. Report completeness checks

In [5]:
required_sections = [
    "Index Result",
    "Sample Context",
    "Top Model Drivers",
    "Interpretation Guardrails",
    "Validation Snapshot",
    "Limitations",
]
report_checks = pd.DataFrame({
    "section": required_sections,
    "present": [section in report_md for section in required_sections],
})
report_checks.to_csv(REPORTS / "report_completeness_checks.csv", index=False)
report_checks

,section,present
0,Index Result,True
1,Sample Context,True
2,Top Model Drivers,True
3,Interpretation Guardrails,True
4,Validation Snapshot,True
5,Limitations,True


## 5. What this notebook demonstrates

This notebook completes the clinical AI product loop: model output → index → confidence → explanation → clinician report → audit trail. It is intentionally designed as decision support, not autonomous diagnosis.